<a href="https://colab.research.google.com/github/MD2001/Self_traine_RAG_Model/blob/main/RAG_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [40]:
!nvidia-smi

Tue May 19 11:27:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P0             29W /   70W |     239MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [41]:
!pip install pymupdf bitsandbytes spacy sentence-transformers torch numpy pandas tqdm requests
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 91.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [42]:
import os
import requests

url = r"https://arxiv.org/pdf/2412.19437"
pdf_path = os.path.join(os.getcwd(), "Data_file.pdf")

if not os.path.exists(pdf_path):
    response = requests.get(url)
    if response.status_code == 200:
        with open(pdf_path, "wb") as f:
            f.write(response.content)
        print(f"[INFO] the file is downloaded at : {pdf_path}")
else:
    print(f"[INFO] file exist : {pdf_path} ")


[INFO] file exist : /content/Data_file.pdf 


In [43]:
import fitz  # PyMuPDF

def clean_text(text: str):
    text = text.replace("\n", " ").strip()
    return text

def open_pdf(file_path: str):
    document = fitz.open(file_path)
    pages_and_texts = []
    for page_num, page in enumerate(document):
        text = page.get_text()
        cleaned_text = clean_text(text)
        pages_and_texts.append({
            "text_of_page": cleaned_text,
            "page_count_char": len(cleaned_text),
            "page_count_word": len(cleaned_text.split(" ")),
            "page_count_sentences": len(cleaned_text.split(". ")),
            "tokens_num": len(cleaned_text) // 4,
            "page_number": page_num + 1
        })
    return pages_and_texts

pages_and_texts = open_pdf("Data_file.pdf")
pages_and_texts[3]


{'text_of_page': '1. Introduction In recent years, Large Language Models (LLMs) have been undergoing rapid iteration and evolution (Anthropic, 2024; Google, 2024; OpenAI, 2024a), progressively diminishing the gap to- wards Artificial General Intelligence (AGI). Beyond closed-source models, open-source models, including DeepSeek series (DeepSeek-AI, 2024a,b,c; Guo et al., 2024), LLaMA series (AI@Meta, 2024a,b; Touvron et al., 2023a,b), Qwen series (Qwen, 2023, 2024a,b), and Mistral series (Jiang et al., 2023; Mistral, 2024), are also making significant strides, endeavoring to close the gap with their closed-source counterparts. To further push the boundaries of open-source model capa- bilities, we scale up our models and introduce DeepSeek-V3, a large Mixture-of-Experts (MoE) model with 671B parameters, of which 37B are activated for each token. With a forward-looking perspective, we consistently strive for strong model performance and economical costs. Therefore, in terms of architectu

In [44]:
import pandas as pd
df = pd.DataFrame(pages_and_texts)
df.head(10)


,text_of_page,page_count_char,page_count_word,page_count_sentences,tokens_num,page_number
0,DeepSeek-V3 Technical Report DeepSeek-AI resea...,1817,238,11,454,1
1,Contents 1 Introduction 4 2 Architecture 6 2.1...,2634,919,766,658,2
2,4.5.3 Batch-Wise Load Balance VS. Sequence-Wis...,1715,572,462,428,3
3,"1. Introduction In recent years, Large Languag...",4248,594,26,1062,4
4,Training Costs Pre-Training Context Extension ...,3287,468,21,821,5
5,verification and reflection patterns of R1 int...,3475,457,23,868,6
6,… Router Input Hidden 𝐮𝐮𝑡𝑡 Output Hidden 𝐡𝐡𝑡𝑡 ...,1557,262,8,389,7
7,where c𝐾𝑉 𝑡 ∈R𝑑𝑐is the compressed latent vecto...,2236,353,9,559,8
8,where 𝑁𝑠and 𝑁𝑟denote the numbers of shared exp...,2901,455,17,725,9
9,Main Model (Next Token Prediction) Embedding L...,2904,451,23,726,10


In [45]:
df.describe().round(2)


,page_count_char,page_count_word,page_count_sentences,tokens_num,page_number
count,53.00,53.00,53.00,53.00,53.00
mean,2836.06,474.38,60.85,708.68,27.00
std,799.86,178.18,124.63,199.99,15.44
min,699.00,99.00,1.00,174.00,1.00
25%,2632.00,386.00,17.00,658.00,14.00
50%,2992.00,457.00,23.00,748.00,27.00
75%,3281.00,516.00,29.00,820.00,40.00
max,4248.00,919.00,766.00,1062.00,53.00


# Turn text to sentences

In [46]:
from spacy.lang.en import English

nlp = English()
nlp.add_pipe("sentencizer")

semtemces = "this my name . hellow mr.ahmed . what is your name ?"
doc = nlp(semtemces)
list(doc.sents)


[this my name ., hellow mr.ahmed ., what is your name ?]

# Now try the `data`

In [47]:
for page in pages_and_texts:
    page["sentences"] = list(nlp(page["text_of_page"]).sents)
    page["sentences_num"] = len(page["sentences"])

for page in pages_and_texts:
    page["sentences"] = [str(s) for s in page["sentences"]]

pages_and_texts[5]


{'text_of_page': 'verification and reflection patterns of R1 into DeepSeek-V3 and notably improves its reasoning performance. Meanwhile, we also maintain control over the output style and length of DeepSeek-V3. Summary of Core Evaluation Results • Knowledge: (1) On educational benchmarks such as MMLU, MMLU-Pro, and GPQA, DeepSeek-V3 outperforms all other open-source models, achieving 88.5 on MMLU, 75.9 on MMLU-Pro, and 59.1 on GPQA. Its performance is comparable to leading closed-source models like GPT-4o and Claude-Sonnet-3.5, narrowing the gap between open-source and closed-source models in this domain. (2) For factuality benchmarks, DeepSeek-V3 demonstrates superior performance among open-source models on both SimpleQA and Chinese SimpleQA. While it trails behind GPT-4o and Claude-Sonnet-3.5 in English factual knowledge (SimpleQA), it surpasses these models in Chinese factual knowledge (Chinese SimpleQA), highlighting its strength in Chinese factual knowledge. • Code, Math, and Reas

In [48]:
pages_and_texts[5]["sentences"][6]


'Code, Math, and Reasoning: (1) DeepSeek-V3 achieves state-of-the-art performance on math-related benchmarks among all non-long-CoT open-source and closed-source models.'

In [49]:
df = pd.DataFrame(pages_and_texts)
df.describe().round(2)


,page_count_char,page_count_word,page_count_sentences,tokens_num,page_number,sentences_num
count,53.00,53.00,53.00,53.00,53.00,53.00
mean,2836.06,474.38,60.85,708.68,27.00,23.94
std,799.86,178.18,124.63,199.99,15.44,14.44
min,699.00,99.00,1.00,174.00,1.00,1.00
25%,2632.00,386.00,17.00,658.00,14.00,16.00
50%,2992.00,457.00,23.00,748.00,27.00,23.00
75%,3281.00,516.00,29.00,820.00,40.00,30.00
max,4248.00,919.00,766.00,1062.00,53.00,54.00


In [50]:
test = list(range(1, 25))
slice_size = 10
x = [test[i:i+slice_size] for i in range(0, len(test), slice_size)]
x


[[1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
 [11, 12, 13, 14, 15, 16, 17, 18, 19, 20],
 [21, 22, 23, 24]]

In [51]:
def chunker(input_list: list[str], slice_size: int = slice_size) -> list[list[str]]:
    return [input_list[i:i+slice_size] for i in range(0, len(input_list), slice_size)]

for page in pages_and_texts:
    page["sentences_chunk"] = chunker(page["sentences"])
    page["sentences_chunk_num"] = len(page["sentences_chunk"])

pages_and_texts[5]


{'text_of_page': 'verification and reflection patterns of R1 into DeepSeek-V3 and notably improves its reasoning performance. Meanwhile, we also maintain control over the output style and length of DeepSeek-V3. Summary of Core Evaluation Results • Knowledge: (1) On educational benchmarks such as MMLU, MMLU-Pro, and GPQA, DeepSeek-V3 outperforms all other open-source models, achieving 88.5 on MMLU, 75.9 on MMLU-Pro, and 59.1 on GPQA. Its performance is comparable to leading closed-source models like GPT-4o and Claude-Sonnet-3.5, narrowing the gap between open-source and closed-source models in this domain. (2) For factuality benchmarks, DeepSeek-V3 demonstrates superior performance among open-source models on both SimpleQA and Chinese SimpleQA. While it trails behind GPT-4o and Claude-Sonnet-3.5 in English factual knowledge (SimpleQA), it surpasses these models in Chinese factual knowledge (Chinese SimpleQA), highlighting its strength in Chinese factual knowledge. • Code, Math, and Reas

In [52]:
df = pd.DataFrame(pages_and_texts)
df.describe().round(2)


,page_count_char,page_count_word,page_count_sentences,tokens_num,page_number,sentences_num,sentences_chunk_num
count,53.00,53.00,53.00,53.00,53.00,53.00,53.00
mean,2836.06,474.38,60.85,708.68,27.00,23.94,2.89
std,799.86,178.18,124.63,199.99,15.44,14.44,1.40
min,699.00,99.00,1.00,174.00,1.00,1.00,1.00
25%,2632.00,386.00,17.00,658.00,14.00,16.00,2.00
50%,2992.00,457.00,23.00,748.00,27.00,23.00,3.00
75%,3281.00,516.00,29.00,820.00,40.00,30.00,3.00
max,4248.00,919.00,766.00,1062.00,53.00,54.00,6.00


In [53]:
pages_and_chunk = []
for page in pages_and_texts:
    for chunk in page["sentences_chunk"]:
        chunk_dict = {}
        paragragh = "".join(chunk).replace("  ", " ").strip()
        chunk_dict["paragraphe"] = paragragh
        chunk_dict["word_count"] = len(paragragh.split(" "))
        chunk_dict["char_count"] = len(paragragh)
        chunk_dict["tokens_count"] = len(paragragh) // 4
        chunk_dict["sentence_count"] = len(chunk)
        chunk_dict["page_number"] = page["page_number"]
        pages_and_chunk.append(chunk_dict)

pages_and_chunk[5], len(pages_and_chunk), len(pages_and_texts)


({'paragraphe': '4.5.3 Batch-Wise Load Balance VS.Sequence-Wise Load Balance . . . . . . . . .27 5 Post-Training 28 5.1 Supervised Fine-Tuning . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .28 5.2 Reinforcement Learning . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .29 5.2.1 Reward Model . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .29 5.2.2 Group Relative Policy Optimization . . . . . . . . . . . . . . . . . . . . . .30 5.3 Evaluations . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .30 5.3.1 Evaluation Settings . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .30 5.3.2 Standard Evaluation . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .31 5.3.3 Open-Ended Evaluation . . . . . . . . . . . . . . . . . . . . . . . . . . . . .',
  'word_count': 298,
  'char_count': 830,
  'tokens_count': 207,
  'sentence_count': 10,
  'page_number': 3},
 153,
 53)

In [54]:
df = pd.DataFrame(pages_and_chunk)
df.describe().round(2)


,word_count,char_count,tokens_count,sentence_count,page_number
count,153.00,153.00,153.00,153.00,153.00
mean,157.03,974.48,243.26,8.29,26.52
std,145.88,584.64,146.14,2.95,13.86
min,1.00,1.00,0.00,1.00,1.00
25%,74.00,572.00,143.00,7.00,15.00
50%,134.00,912.00,228.00,10.00,28.00
75%,191.00,1301.00,325.00,10.00,39.00
max,877.00,2975.00,743.00,10.00,53.00


In [55]:
df[df["word_count"] < 4]


,paragraphe,word_count,char_count,tokens_count,sentence_count,page_number
21,9,1,1,0,1,9


In [56]:
filtered_chunks = [chunk for chunk in pages_and_chunk if chunk["tokens_count"] >= 30]
filtered_chunks[111]


{'paragraphe': 'A study of bfloat16 for deep learning training.arXiv preprint arXiv:1905.12322, 2019.S. Krishna, K. Krishna, A. Mohananey, S. Schwarcz, A. Stambler, S. Upadhyay, and M. Faruqui.Fact, fetch, and reason: A unified evaluation of retrieval-augmented generation.CoRR, abs/2409.12941, 2024.doi: 10.48550/ARXIV.2409.12941.URL https://doi.org/10.485 50/arXiv.2409.12941.T. Kwiatkowski, J. Palomaki, O. Redfield, M. Collins, A. P. Parikh, C. Alberti, D. Epstein, I. Polosukhin, J. Devlin, K. Lee, K. Toutanova, L. Jones, M. Kelcey, M. Chang, A. M. Dai, J. Uszkoreit, Q. Le, and S. Petrov.Natural questions: a benchmark for question answering research.Trans.',
 'word_count': 84,
 'char_count': 648,
 'tokens_count': 162,
 'sentence_count': 10,
 'page_number': 39}

In [57]:
from sentence_transformers import SentenceTransformer, util
model = SentenceTransformer("all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [58]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())


2.10.0+cu128
True


In [59]:
test_set = ["my name", "her name", "the naem"]
embedding = model.encode(test_set)
embedding = dict(zip(test_set, embedding))
embedding, embedding["my name"].shape


({'my name': array([-1.11268573e-01,  3.23448479e-02, -3.83405983e-02,  2.23898515e-02,
         -4.03140448e-02, -5.62008433e-02,  2.33329967e-01,  3.92086506e-02,
         -1.01078916e-02, -6.38170317e-02,  1.64089613e-02, -3.45565118e-02,
          4.02090512e-02, -7.56014213e-02, -8.34885146e-03,  2.65755560e-02,
         -1.42915146e-02,  6.17216621e-03, -8.46135914e-02, -7.72886276e-02,
         -4.17721681e-02,  6.27691001e-02, -3.24081848e-05, -2.02984978e-02,
         -8.92573670e-02,  6.10290561e-03, -3.12835611e-02,  9.11155567e-02,
         -3.69639210e-02, -1.61543220e-01, -3.80761048e-04,  7.48015754e-03,
          9.98783335e-02,  5.72649203e-02, -4.66819946e-03, -4.17357497e-02,
         -1.26731172e-01, -2.11646315e-02,  5.62063605e-02, -1.07810730e-02,
         -4.70561814e-03, -1.19292334e-01,  2.42646094e-02,  5.37946960e-03,
          7.11642625e-03,  7.88037013e-03,  6.28924463e-04,  2.45066918e-02,
          1.61230508e-02,  9.47958529e-02, -9.59324911e-02, -8.88

In [60]:
from tqdm import tqdm

for chunk in tqdm(pages_and_chunk):
    chunk["embedding"] = model.encode(chunk["paragraphe"], device="cuda")


100%|██████████| 153/153 [00:02<00:00, 55.63it/s]


In [61]:
page_and_Embedding_df = pd.DataFrame(pages_and_chunk)
page_and_Embedding_file_name = "page_and_Embedding.csv"
page_and_Embedding_path = os.path.join(os.getcwd(), page_and_Embedding_file_name)
page_and_Embedding_df.to_csv(page_and_Embedding_path, index=False)


In [62]:
pages_and_chunk = pd.read_csv(page_and_Embedding_path)
pages_and_chunk


,paragraphe,word_count,char_count,tokens_count,sentence_count,page_number,embedding
0,DeepSeek-V3 Technical Report DeepSeek-AI resea...,222,1767,441,10,1,[-7.36544952e-02 -5.92406169e-02 5.75507320e-...
1,arXiv:2412.19437v2 [cs.CL] 18 Feb 2025,5,38,9,2,1,[-1.14521943e-01 7.90134817e-03 -7.34652132e-...
2,Contents 1 Introduction 4 2 Architecture 6 2.1...,302,952,238,10,2,[-1.76019520e-02 -5.03365062e-02 -4.91420068e-...
3,14 3.3.1 Mixed Precision Framework . . . . . ....,324,912,228,10,2,[-1.62387546e-02 3.45402360e-02 -1.01859644e-...
4,21 4.2 Hyper-Parameters . . . . . . . . . . . ...,267,742,185,9,2,[ 2.22861618e-02 3.54707390e-02 -1.13855258e-...
...,...,...,...,...,...,...,...
148,1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 1...,877,2961,740,1,49,[-8.76501352e-02 3.37730092e-03 -2.45083366e-...
149,1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 1...,877,2968,742,1,50,[-8.87522548e-02 2.17784266e-03 -2.23974865e-...
150,1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 1...,877,2975,743,1,51,[-8.97150487e-02 9.12359171e-03 -2.74221674e-...
151,1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 1...,877,2975,743,1,52,[-8.86220858e-02 6.90447586e-03 -2.55782306e-...


In [63]:
data = pages_and_chunk.to_dict(orient="records")
data[5]


{'paragraphe': '4.5.3 Batch-Wise Load Balance VS.Sequence-Wise Load Balance . . . . . . . . .27 5 Post-Training 28 5.1 Supervised Fine-Tuning . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .28 5.2 Reinforcement Learning . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .29 5.2.1 Reward Model . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .29 5.2.2 Group Relative Policy Optimization . . . . . . . . . . . . . . . . . . . . . .30 5.3 Evaluations . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .30 5.3.1 Evaluation Settings . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .30 5.3.2 Standard Evaluation . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .31 5.3.3 Open-Ended Evaluation . . . . . . . . . . . . . . . . . . . . . . . . . . . . .',
 'word_count': 298,
 'char_count': 830,
 'tokens_count': 207,
 'sentence_count': 10,
 'page_number': 3,
 'embedding': '[-5.76343276e-02  1.86269917e-02

In [64]:
import torch
import numpy as np
device = "cuda" if torch.cuda.is_available() else "cpu"
device


'cuda'

In [65]:
for item in data:
    item['embedding'] = np.fromstring(item['embedding'].strip('[]'), sep=' ')


In [66]:
embeddings = np.array([item["embedding"] for item in data])
embeddings_df = pd.DataFrame(embeddings)
print(embeddings_df.shape)  # should be (153, 384)


(153, 384)


153 rows x 384 dimensions — one embedding vector per chunk.

In [67]:
from time import perf_counter as timer

query = "attentions in models datas"
embeddings_tensor = torch.tensor(np.array(embeddings), dtype=torch.float32)
query_embedding = model.encode(query, device=device)

start = timer()
dot_product = util.dot_score(a=query_embedding, b=embeddings_tensor)[0]
end = timer()
print(f"[INFO]: time it take {(end-start):.5f} seconds")

dot_topk = torch.topk(dot_product, 3)
dot_topk


[INFO]: time it take 0.00033 seconds


torch.return_types.topk(
values=tensor([0.4049, 0.3879, 0.3805]),
indices=tensor([113, 119,  17]))

In [68]:
# just test what idx.item() do
[idx.item() for idx in dot_topk.indices]

#it turn tensor to scaller

[113, 119, 17]

In [69]:
# Show the top-k matching paragraphs
for score, idx in zip(dot_topk.values, dot_topk.indices):
    print(f"Score: {score:.4f} | Page: {data[idx.item()]['page_number']}")
    print(data[idx.item()]['paragraphe'])
    print("-" * 120)


Score: 0.4049 | Page: 38
In The Thirty-eighth Annual Conference on Neural Information Processing Systems.Y. He, S. Li, J. Liu, Y. Tan, W. Wang, H. Huang, X. Bu, H. Guo, C. Hu, B. Zheng, et al.Chi- nese simpleqa: A chinese factuality evaluation for large language models.arXiv preprint arXiv:2411.07140, 2024.D. Hendrycks, C. Burns, S. Basart, A. Zou, M. Mazeika, D. Song, and J. Steinhardt.Measuring massive multitask language understanding.arXiv preprint arXiv:2009.03300, 2020.38
------------------------------------------------------------------------------------------------------------------------
Score: 0.3879 | Page: 40
Y. Leviathan, M. Kalman, and Y. Matias.Fast inference from transformers via speculative decoding.In International Conference on Machine Learning, ICML 2023, 23-29 July 2023, Honolulu, Hawaii, USA, volume 202 of Proceedings of Machine Learning Research, pages 19274–19286.PMLR, 2023.URL https://proceedings.mlr.press/v202/leviathan23 a.html.H. Li, Y. Zhang, F. Koto, Y. Yan

# functionalize the query search

In [70]:
def get_answer(query:str,
               embedings_data=embeddings,
               embedding_model: SentenceTransformer=model,
               top_k :int = 5,
               time_taken:bool =True):
    # embed the query
    query_embed= embedding_model.encode(query,convert_to_tensor=True)

    # make sure that data are in the same device (cuda) the GPU in this case
    embedings_data = torch.tensor(np.array(embedings_data), dtype=torch.float32).to('cuda')

    # compare the embedding of query with ather embeddings frome book/dataset
    start = timer()
    dot_product = util.dot_score(a=query_embed, b=embedings_data)[0]
    end = timer()

    if time_taken:
        print(f"[INFO]time {(end-start):.5f}")
    top_results = torch.topk(dot_product,top_k)
    scores=[score.item() for score in top_results[0]]
    indeces =[score.item() for score in top_results[1]]
    # scores,indeces = top_results
    # print(f"{top_results}")
    return scores,indeces

def print_top_Result(query:str,
               embedings_data=embeddings,
               embedding_model: SentenceTransformer=model,
               top_k :int = 5,
               time_taken:bool =True,
               pages_and_chunk=data):
    score,indcis=get_answer(query,embedings_data,embedding_model,top_k,time_taken)

    for s, i in zip(score, indcis):
        print(f"Score: {s:.4f} | Index: {i} | Page_number: {pages_and_chunk[i]['page_number']}\nChunk: {pages_and_chunk[i]["paragraphe"]}")
        print("-"*120)

In [71]:
print_top_Result("names")

[INFO]time 0.00011
Score: 0.2919 | Index: 143 | Page_number: 45
Chunk: Appendix A. Contributions and Acknowledgments Research & Engineering Aixin Liu Bing Xue Bingxuan Wang Bochao Wu Chengda Lu Chenggang Zhao Chengqi Deng Chenyu Zhang* Chong Ruan Damai Dai Daya Guo Dejian Yang Deli Chen Erhang Li Fangyun Lin Fucong Dai Fuli Luo* Guangbo Hao Guanting Chen Guowei Li H. Zhang Han Bao* Hanwei Xu Haocheng Wang* Haowei Zhang Honghui Ding Huajian Xin* Huazuo Gao Hui Qu Jianzhong Guo Jiashi Li Jiawei Wang* Jingchang Chen Jingyang Yuan Junjie Qiu Junlong Li Junxiao Song Kai Dong Kai Hu* Kaige Gao Kang Guan Kexin Huang Kuai Yu Lean Wang Lecong Zhang Liang Zhao Litong Wang Liyue Zhang Mingchuan Zhang Minghua Zhang Minghui Tang Panpan Huang Peiyi Wang Qiancheng Wang Qihao Zhu Qinyu Chen Qiushi Du Ruiqi Ge Ruisong Zhang Ruizhe Pan Runji Wang Runxin Xu Ruoyu Zhang Shanghao Lu Shangyan Zhou Shanhuang Chen Shengfeng Ye Shirong Ma Shiyu Wang Shuiping Yu Shunfeng Zhou Shuting Pan Tao Yun Tian Pei Wangdi

In [72]:
# Get GPU available memory
import torch
gpu_memory_bytes = torch.cuda.get_device_properties(0).total_memory
gpu_memory_gb = round(gpu_memory_bytes / (2 ** 30))
print(f"Available GPU memory: {gpu_memory_gb} GB")

Available GPU memory: 15 GB


# This test from tetorial

In [73]:
# Note: the following is Gemma focused, however, there are more and more LLMs of the 2B and 7B size appearing for local use.
if gpu_memory_gb < 5.1:
  print(f"Your available GPU memory is {gpu_memory_gb}GB, you may not have enough memory to run a Gemma LLM locally without quantization.")
elif gpu_memory_gb < 8.1:
  print(f"GPU memory: {gpu_memory_gb} | Recommended model: Gemma 2B in 4-bit precision.")
  use_quantization_config = True
  model_id = "google/gemma-2b-it"
elif gpu_memory_gb < 19.0:
  print(f"GPU memory: {gpu_memory_gb} | Recommended model: Gemma 2B in float16 or Gemma 7B in 4-bit precision.")
  use_quantization_config = False
  model_id = "google/gemma-2b-it"
elif gpu_memory_gb > 19.0:
  print(f"GPU memory: {gpu_memory_gb} | Recommend model: Gemma 7B in 4-bit or float16 precision.")
  use_quantization_config = False
  model_id = "google/gemma-7b-it"

print(f"use_quantization_config set to: {use_quantization_config}")
print(f"model_id set to: {model_id}")

GPU memory: 15 | Recommended model: Gemma 2B in float16 or Gemma 7B in 4-bit precision.
use_quantization_config set to: False
model_id set to: google/gemma-2b-it


In [74]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

In [75]:
import torch
from transformers import AutoTokenizer,AutoModelForCausalLM,BitsAndBytesConfig
from transformers.utils import is_flash_attn_2_available

#configer quntization configruation
quantize_config = BitsAndBytesConfig(load_in_4bit=True,
                                     bnb_4bit_compute_dtype=torch.float16)

if (is_flash_attn_2_available()) and (torch.cuda.get_device_capability()[0] >=8):
  attention_imp = "flash_attention_2"
else :
  attention_imp ="sdpa"

model_id = "google/gemma-2b-it"

#toknization
toknizer= AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_id)